## Configuration Set Up

In [0]:
from pyspark.sql import functions as F
import re

# Current Date
CURRENT_DATE = F.date_format(F.current_date(), "yyyyMMdd")

# Source 
# SOURCE_PATH = f"abfss://raw@bsrtestdatalake.dfs.core.windows.net/audits/{CURRENT_DATE}"
SOURCE_PATH = "abfss://raw@bsrtestdatalake.dfs.core.windows.net/audits/20250523"

# Destination
CATALOG = "devs"
SCHEMA  = "bronze_william_lyagoba1" #change when when added to git
CSV_OPTIONS = {
    "header":          "true",
    "inferSchema":     "true",
    "encoding":        "UTF-8",
    "sep":             ",",
    "escape":          '"',
    "multiLine":       "false",
    "timestampFormat": "yyyy-MM-dd HH:mm:ss",
}

print(f"Source  : {SOURCE_PATH}")
print(f"Target  : {CATALOG}.{SCHEMA}.<table_per_file>")

Source  : abfss://raw@bsrtestdatalake.dfs.core.windows.net/audits/20250523
Target  : devs.bronze_william_lyagoba1.<table_per_file>


## Check bronze schema exists

In [0]:
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA} "
)
print(f"Schema {CATALOG}.{SCHEMA} ready.")

Schema devs.bronze_william_lyagoba1 ready.


## List and Load files

In [0]:
# List files — mirrors your existing code
files = dbutils.fs.ls(SOURCE_PATH)
for file in files:
    df = spark.read.csv(file.path, header=True, inferSchema=True)

    # derive table name from filename  e.g. "Audit Results.csv" → "audit_results"
    table_name = re.sub(r"[^a-z0-9]+", "_", file.name.lower().replace(".csv", "")).strip("_")

    (
        df
        .withColumn("_source_file_name",    F.lit(file.name))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}")
    )

    print(f"Loaded {file.name} to {CATALOG}.{SCHEMA}.{table_name}")

Loaded audit_episodes to devs.bronze_william_lyagoba1.audit_episodes
Loaded audit_subjects to devs.bronze_william_lyagoba1.audit_subjects
